In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

def simuliere_ztil_realistisch(staats_angriff, angriffs_zeitpunkt, angriffs_typ, sperr_haerte, ausfall_rate, latenz_sync, rausch_faktor):
    np.random.seed(42)
    zeit = np.linspace(0, 200, 200)
    ehrliche_minderheit = np.ones(200) * 100.0
    staats_kollektiv = np.ones(200) * 100.0

    if staats_angriff:
        for i, t in enumerate(zeit):
            if t >= angriffs_zeitpunkt:
                dt = t - angriffs_zeitpunkt
                rauschen_ehr = np.random.normal(0, rausch_faktor * 3)
                rauschen_kol = np.random.normal(0, rausch_faktor * 5)
                
                if angriffs_typ == "Plötzlich (Hard-Cut)":
                    staats_kollektiv[i] = max(0, 100.0 - dt * (sperr_haerte * 0.8) + rauschen_kol)
                    if dt < latenz_sync:
                        ehrliche_minderheit[i] = max(40.0, 100.0 - dt * 0.9 + rauschen_ehr)
                    else:
                        ziel_utility = 100.0 * (1.0 - (ausfall_rate / 100.0))
                        ehrliche_minderheit[i] = min(ziel_utility, ehrliche_minderheit[i-1] + 1.5) + rauschen_ehr
                else:
                    if dt < 35:
                        ehrliche_minderheit[i] = max(50.0, 100.0 - (dt * 0.9) + rauschen_ehr)
                        staats_kollektiv[i] = max(10.0, 100.0 - (dt * 1.2) + rauschen_kol)
                    else:
                        if dt < (35 + latenz_sync):
                            ehrliche_minderheit[i] = max(30.0, ehrliche_minderheit[i] - 0.6 + rauschen_ehr)
                        else:
                            ziel_utility = 100.0 * (1.0 - (ausfall_rate / 100.0))
                            ehrliche_minderheit[i] = min(ziel_utility, ehrliche_minderheit[i-1] + 1.2) + rauschen_ehr
                        staats_kollektiv[i] = max(0, staats_kollektiv[i-1] - (sperr_haerte * 1.5) + rauschen_kol)
    else:
        ehrliche_minderheit += np.random.normal(0, rausch_faktor, size=200)
        staats_kollektiv += np.random.normal(0, rausch_faktor, size=200)

    plt.figure(figsize=(10, 5))
    plt.plot(zeit, ehrliche_minderheit, label="Ehrliche Minderheit (Real-Raum mit Ausfällen)", color="#00ffcc", linewidth=2.0)
    plt.plot(zeit, staats_kollektiv, label=f"Kollektiv ({angriffs_typ})", color="#ff3366", linestyle="--", linewidth=2.0)

    if staats_angriff:
        plt.axvline(x=angriffs_zeitpunkt, color="yellow", linestyle=":", label=f"Angriff / Trigger bei t={angriffs_zeitpunkt}")

    plt.title("HP-v10: Realistische ZTIL-Resilienz unter Stochastik & Infrastruktur-Schocks", fontsize=12, color="white", pad=15)
    plt.xlabel("Zeitverlauf t (System-Ticks)", fontsize=11, color="white")
    plt.ylabel("Effektive Sub-Grid Utility", fontsize=11, color="white")
    plt.grid(True, linestyle="--", alpha=0.3)
    plt.ylim(-15, 120)
    plt.legend(loc="lower left")
    plt.show()

interact(
    simuliere_ztil_realistisch,
    staats_angriff=widgets.Checkbox(value=True, description="Angriff aktiv:"),
    angriffs_zeitpunkt=widgets.IntSlider(min=20, max=140, step=10, value=50, description="Angriffs-Tick:"),
    angriffs_typ=widgets.Dropdown(options=["Plötzlich (Hard-Cut)", "Schleichend (Slow Poisoning)"], value="Plötzlich (Hard-Cut)", description="Vektor:"),
    sperr_haerte=widgets.FloatSlider(min=1.0, max=10.0, step=0.5, value=6.0, description="Sperr-Härte:"),
    ausfall_rate=widgets.FloatSlider(min=0.0, max=40.0, step=5.0, value=15.0, description="Ausfall-Rate (%):"),
    latenz_sync=widgets.IntSlider(min=5, max=50, step=5, value=20, description="Sync-Latenz:"),
    rausch_faktor=widgets.FloatSlider(min=0.0, max=3.0, step=0.5, value=1.0, description="Infrastruktur-Noise:")
)